# watermark removal testing on Kaggle (2x T4)

Notebook settings: **Accelerator = GPU T4 x2**, **Internet = On**.

Add `wmr-0.2.0.zip` to the notebook as a dataset (or upload it to `/kaggle/working`), then run the cells in order.

In [3]:
import glob, os, shutil, zipfile
ROOT = '/kaggle/working/wmr-0.2.0'
if not os.path.exists(ROOT):
    zips = glob.glob('/kaggle/input/**/wmr-0.2.0.zip', recursive=True) + glob.glob('/kaggle/working/wmr-0.2.0.zip')
    dirs = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/scripts/setup_kaggle.sh', recursive=True)]
    if zips:
        zipfile.ZipFile(zips[0]).extractall('/kaggle/working')
    elif dirs:                       # Kaggle unpacked the zip for us
        shutil.copytree(dirs[0], ROOT)
    else:
        raise SystemExit('wmr-0.2.0.zip not found: add it as a dataset or upload it to /kaggle/working')
print(os.listdir(ROOT))

['notebooks', 'wmr-0.2.0-py3-none-any.whl', 'scripts', 'README.md', 'requirements', 'wmr', 'pyproject.toml', 'tests']


In [5]:
!bash /kaggle/working/wmr-0.2.0/scripts/setup_kaggle.sh

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
{
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "python": "3.13.16",
  "machine": "x86_64",
  "torch_device": "cuda",
  "gpus": [
    0,
    1
  ],
  "torch": "2.11.0+cu130",
  "gpu_names": [
    "Tesla T4",
    "Tesla T4"
  ],
  "ort_providers": [
    "TensorrtExecutionProvider",
    "CUDAExecutionProvider",
    "CPUExecutionProvider"
  ],
  "av": "19.0.2",
  "cv2": "4.14.0",
  "numpy": "2.1.3",
  "scipy": "1.16.3",
  "onnxruntime": "1.31.1"
}


## 1. Self-test on a synthetic clip

Builds a clip with three known overlays, removes them, and scores the result against the clean original.
`frames_with_mask_recall_below_95pct` should be 0 and `psnr_in_mask_after` well above `psnr_in_mask_before`.

In [34]:
%cd /kaggle/working
!python -m wmr synth -o synth1 --size 1280x720 --frames 300 --overlay "@my_channel:bounce:1.0:40" --overlay "SAMPLE MARK:static:0.4:40" --overlay "hello2026:teleport:1.0:40" --dwell 100
!python -m wmr run synth1/marked.mp4 -o synth1/out.mp4 --texts synth1/texts.txt --preview synth1/preview.mp4
!python -m wmr eval synth1 synth1/out.mp4 --tracks synth1/out.tracks.json

/kaggle/working
wrote synth1/marked.mp4, clean.mp4, gt.json, texts.txt
[analyze] marked.mp4 1280x720 @ 30.00 fps, OCR on CPUExecutionProvider, every 10 frames
[analyze] pass A: 300 frames, 87 candidate detections, 3 scene cuts
[analyze] 4 track(s): 'hello2026', 'SAMPLE MARK', '@my_channel', '@my_channel'
  track 0: 'hello2026' [list] teleport, tier=fill, frames 0-299 (300), conf 0.80, opacity 0.94
  track 1: 'SAMPLE MARK' [list] static, tier=alpha, frames 0-299 (300), conf 0.64, opacity 0.43
  track 2: '@my_channel' [list] bounce, tier=fill, frames 0-299 (300), conf 0.93, opacity n/a
  track 3: '@my_channel' [list] bounce, tier=fill, frames 0-299 (300), conf 0.82, opacity n/a
^C
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.13/dist-packages/wmr/__main__.py", line 3, in <module>
    main()
    ~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/wmr/cli.py", lin

## 2. Your videos

Put the videos in a dataset and list the overlay strings, one per line. Leave `TEXTS` empty to use discovery only.

In [33]:
IN_DIR  = '/kaggle/input/datasets/yxyxtx/video-test-set-1'          # folder with your videos
OUT_DIR = '/kaggle/working/out'
TEXTS = ''
os.makedirs(OUT_DIR, exist_ok=True)
open('/kaggle/working/texts.txt', 'w').write(TEXTS + '\n')
print(sorted(os.listdir(IN_DIR))[:20])

['BJ  1.mp4', 'BJ  10.mp4', 'BJ  11.mp4', 'BJ  12.mp4', 'BJ  13.mp4', 'BJ  14.mp4', 'BJ  15.mp4', 'BJ  16.mp4', 'BJ  17.mp4', 'BJ  18.mp4', 'BJ  19.mp4', 'BJ  2.mp4', 'BJ  20.mp4', 'BJ  3.mp4', 'BJ  4.mp4', 'BJ  5.mp4', 'BJ  6.mp4', 'BJ  7.mp4', 'BJ  8.mp4', 'BJ  9.mp4']


### One video first: analyze and look at the preview

Red boxes will be removed. Yellow boxes were discovered but need your approval (set `"enabled": true` in the tracks file).

In [ ]:
VIDEO = os.path.join(IN_DIR, sorted(os.listdir(IN_DIR))[0])
!python -m wmr analyze "{VIDEO}" -o /kaggle/working/one.tracks.json --texts /kaggle/working/texts.txt --discover --preview /kaggle/working/one_preview.mp4
from IPython.display import Video
Video('/kaggle/working/one_preview.mp4', embed=True, width=720)

In [ ]:
!python -m wmr remove "{VIDEO}" /kaggle/working/one.tracks.json -o /kaggle/working/one_out.mp4
!python -m wmr verify /kaggle/working/one_out.mp4 --tracks /kaggle/working/one.tracks.json
Video('/kaggle/working/one_out.mp4', embed=True, width=720)

### The whole folder, one worker per GPU

In [ ]:
!python -m wmr batch "{IN_DIR}" -o "{OUT_DIR}" --texts /kaggle/working/texts.txt --discover --gpus 0,1
import json
for f in sorted(glob.glob(OUT_DIR + '/*.report.json')):
    r = json.load(open(f)); v = r.get('verify', {})
    print(os.path.basename(r['output']), '| leaked frames:', v.get('leak_frames'), 'of', v.get('checked_frames'), '| borrowed share:', r['remove'].get('borrowed_share'))

## 3. Optional: ProPainter engine for static opaque overlays

Higher quality where the background under the text is never visible. It uses both GPUs for a single video.
ProPainter is under the NTU S-Lab License 1.0 (non-commercial). This path has not been run by the author yet:
if a job fails, its frames fall back to the built-in fill and the error is saved next to the job.

In [ ]:
!bash /kaggle/working/wmr-0.2.0/scripts/setup_kaggle.sh propainter
!python -m wmr remove "{VIDEO}" /kaggle/working/one.tracks.json -o /kaggle/working/one_out_pp.mp4 --engine propainter --gpus 0,1 --keep-work
Video('/kaggle/working/one_out_pp.mp4', embed=True, width=720)